# Linear Regression with Batch Gradient Descent

A small, reproducible implementation of a gradient-descent regressor. This notebook builds the model from first principles, fits it to synthetic data, and checks both its predictions and its convergence.

**Requirements:** Python 3.10+, NumPy, and Matplotlib.

## Learning objectives

- Define the mean-squared-error objective for linear regression.
- Implement batch gradient descent for one or more numeric features.
- Track training loss and inspect the fitted line and convergence.

The model includes an intercept and uses the full training set for each gradient update. Features are not scaled automatically, so the learning rate should be chosen with their scale in mind.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Create a reproducible dataset

We generate observations from a known linear relationship with a small amount of random noise. Keeping the true parameters lets us check whether training recovers a sensible model.

In [ ]:
rng = np.random.default_rng(seed=42)
X = rng.uniform(-3.0, 3.0, size=(200, 1))
true_slope = 4.2
true_intercept = 2.5
y = true_slope * X[:, 0] + true_intercept + rng.normal(0.0, 1.0, size=X.shape[0])

print(f"Observations: {X.shape[0]}")
print(f"Features:     {X.shape[1]}")
print(f"Feature range: [{X.min():.2f}, {X.max():.2f}]")

## 2. Implement the regressor

For predictions `y_hat = X @ coef + intercept`, the mean squared error is the average squared residual. Each iteration updates the coefficients and intercept in the direction that reduces this loss. Inputs are checked up front so common shape and value errors fail with a useful message.

In [ ]:
class GDRegressor:
    """Linear regression trained with batch gradient descent."""

    def __init__(self, epochs: int = 10_000, learning_rate: float = 0.1) -> None:
        if isinstance(epochs, bool) or not isinstance(epochs, (int, np.integer)) or epochs <= 0:
            raise ValueError("epochs must be a positive integer")
        if not np.isfinite(learning_rate) or learning_rate <= 0:
            raise ValueError("learning_rate must be a finite positive number")

        self.epochs = int(epochs)
        self.learning_rate = float(learning_rate)
        self.coef_: np.ndarray | None = None
        self.intercept_: float | None = None
        self.loss_history_: list[float] = []

    @staticmethod
    def _as_feature_matrix(X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
            raise ValueError("X must be a non-empty 1D or 2D feature array")
        if not np.all(np.isfinite(X)):
            raise ValueError("X must contain only finite values")
        return X

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GDRegressor":
        X = self._as_feature_matrix(X)
        y = np.asarray(y, dtype=float)
        if y.ndim != 1:
            raise ValueError("y must be a one-dimensional target array")
        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y must contain the same number of observations")
        if not np.all(np.isfinite(y)):
            raise ValueError("y must contain only finite values")

        n_samples, n_features = X.shape
        coef = np.zeros(n_features, dtype=float)
        intercept = 0.0
        self.loss_history_ = []

        for _ in range(self.epochs):
            residuals = X @ coef + intercept - y
            coef -= self.learning_rate * (2.0 / n_samples) * (X.T @ residuals)
            intercept -= self.learning_rate * (2.0 / n_samples) * residuals.sum()

            updated_residuals = X @ coef + intercept - y
            self.loss_history_.append(float(np.mean(updated_residuals**2)))

        self.coef_ = coef
        self.intercept_ = float(intercept)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.coef_ is None or self.intercept_ is None:
            raise RuntimeError("Call fit before predict")
        X = self._as_feature_matrix(X)
        if X.shape[1] != self.coef_.shape[0]:
            raise ValueError("X must have the same number of features used during fit")
        return X @ self.coef_ + self.intercept_

## 3. Fit the model

The following settings are suitable for this small dataset. The estimator can also be initialized with the same keyword arguments as the original example: `GDRegressor(epochs=10_000, learning_rate=0.1)`.

In [ ]:
model = GDRegressor(epochs=10_000, learning_rate=0.1)
model.fit(X, y)

print(f"Estimated slope:     {model.coef_[0]:.3f} (true value: {true_slope:.3f})")
print(f"Estimated intercept: {model.intercept_:.3f} (true value: {true_intercept:.3f})")
print(f"Final training MSE:  {model.loss_history_[-1]:.3f}")

## 4. Inspect predictions

Plotting the observed data and fitted line gives a quick visual check that the model captures the trend.

In [ ]:
X_line = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
y_line = model.predict(X_line)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X[:, 0], y, alpha=0.65, label="Observed data")
ax.plot(X_line[:, 0], y_line, color="darkorange", linewidth=2.5, label="Gradient-descent fit")
ax.set(title="Linear Regression Fit", xlabel="Feature", ylabel="Target")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Evaluate and inspect convergence

The metrics below are calculated on the same observations used for fitting, so they describe training performance rather than out-of-sample generalization. The loss curve helps identify whether optimization is still making progress or has stabilized.

In [ ]:
predictions = model.predict(X)
residuals = y - predictions
mae = np.mean(np.abs(residuals))
rmse = np.sqrt(np.mean(residuals**2))

print(f"Training MAE:  {mae:.3f}")
print(f"Training RMSE: {rmse:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model.loss_history_, color="steelblue")
ax.set(title="Training Loss by Epoch", xlabel="Epoch", ylabel="Mean squared error")
fig.tight_layout()
plt.show()

## Key takeaways

- Batch gradient descent updates parameters using gradients calculated from all observations.
- The learning rate and feature scale affect whether training converges; scale features when working with features on very different ranges.
- A low training loss alone does not measure generalization. For real applications, reserve a validation or test set.

### Next steps

Try changing the learning rate, adding more input features, or splitting the data into training and test sets. Compare the results with a trusted library implementation when applying the method to a real dataset.